## Step 1: Installation
Install the core LlamaIndex library along with the specific integrations for Ollama LLMs and
embeddings 1-3.

## Step 2: Setup Local Models with Ollama
LlamaIndex uses a global Settings object to manage the LLM and embedding models used across the application. By using Ollama, we can run models like llama3.2 

In [25]:
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings

In [26]:
# Using llama3.2 locally ensures data privacy and zero cost per token.
Settings.llm = Ollama(model="qwen2.5:3b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text:latest")

## Step 3: Data Ingestion
Before querying, you must load your private data. SimpleDirectoryReader is a versatile data connector that can ingest various formats (PDFs, text, etc.) from a local folder

In [27]:
from llama_index.core import SimpleDirectoryReader
import os

os.makedirs("data", exist_ok=True)

documents = SimpleDirectoryReader("./data", filename_as_id=True).load_data()
print(f"Loaded {len(documents)} document(s).")

Loaded 50 document(s).


In [28]:
for i, doc in enumerate(documents[:5]):  # show only first 5
    print(f"{i}: {doc.metadata.get('file_name')} ({doc.metadata.get('file_path')})")
    print(doc.text[:75])
    print("-" * 60)

0: Cover Letter 2026-01-28.pdf (/home/pawankrgunjan/Development/GenerativeAI/LlamaIndex/data/Cover Letter 2026-01-28.pdf)
Dear Pawan Kumar,
We are making this offer pursuant to our discussions. We 
------------------------------------------------------------
1: Cover Letter 2026-01-28.pdf (/home/pawankrgunjan/Development/GenerativeAI/LlamaIndex/data/Cover Letter 2026-01-28.pdf)
 
Overview of your offer
Role
AI/ML Computational Science Sr 
Analyst
Organ
------------------------------------------------------------
2: Cover Letter 2026-01-28.pdf (/home/pawankrgunjan/Development/GenerativeAI/LlamaIndex/data/Cover Letter 2026-01-28.pdf)
 
Equity*
Employee Share Purchase Plan
Purchase shares up to 10% of eligibl
------------------------------------------------------------
3: Cover Letter 2026-01-28.pdf (/home/pawankrgunjan/Development/GenerativeAI/LlamaIndex/data/Cover Letter 2026-01-28.pdf)
If you have any questions at all on the offer or need assistance in complet
-------------------------

## Step 4: Indexing
Once loaded, the data must be structured so the LLM can efficiently retrieve relevant context. A VectorStoreIndex chunks the text and stores it as searchable embeddings

In [29]:
from llama_index.core import VectorStoreIndex

# Concept: A Data Index structures your data into a representation 
# that is performant for LLMs to consume [10].
# It handles chunking and embedding the documents automatically [13].
index = VectorStoreIndex.from_documents(documents)

print("Index created successfully.")

2026-02-08 20:41:27,183 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:29,869 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:32,781 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:35,373 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:38,478 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:42,206 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:41:42,329 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Index created successfully.


In [30]:
ref = getattr(index, "ref_doc_info", None)
print("ref_doc_info available?", ref is not None)

if ref:
    for doc_id, info in ref.items():
        print("doc_id:", doc_id, "num_nodes:", len(info.node_ids))
        break

ref_doc_info available? True
doc_id: /home/pawankrgunjan/Development/GenerativeAI/LlamaIndex/data/Cover Letter 2026-01-28.pdf_part_0 num_nodes: 1


#### How many nodes/chunks exist?

In [31]:
docstore = index.storage_context.docstore
print("Nodes in docstore:", len(docstore.docs))

Nodes in docstore: 61


#### Retrieval test (do we get relevant chunks back?)

In [32]:
retriever = index.as_retriever(similarity_top_k=3)
hits = retriever.retrieve("What is this data about?")
for h in hits:
    print("score:", h.score, "file:", h.node.metadata.get("file_name"))
    print(h.node.get_content()[:200])
    print("-" * 60)

2026-02-08 20:41:42,583 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


score: 0.5768514425998182 file: Exhibit A - List of Prior Works.pdf
Exhibit A – List of Prior Works 
This document will be considered as part and parcel of the Terms of Employment. 
S.No List of Prior Works 
ACKNOWLEDGED AND AGREED: 
______________________________ 
Na
------------------------------------------------------------
score: 0.5703835182021717 file: Review Documents for Offer for Job Application Pawan Kumar Gunjan _- AIOC-S01589417 AIML Computational Science Sr Analyst (C07995484).pdf
The Company at its sole discretion (including 
but not limited to unforeseen circumstances like a pandemic or natural calamities) may extend or defer the start date of your 
joining, for which deferme
------------------------------------------------------------
score: 0.5623772906307655 file: Performance Cycle.pdf
Communication on performance, career progression and the variable bonus payout process for 
employees joining the India Operations from Levels 10 to 13. 
 
1. Accenture Operations is a

## Step 5: Querying
Finally, you can create a QueryEngine to interact with your data. This engine retrieves the most relevant snippets from your index and provides them to the LLM to generate an answer

In [33]:
# Concept: A Query Engine is a natural language interface for question-answering [14].
# It combines the retrieval of relevant context with LLM generation [10, 13].
query_engine = index.as_query_engine()

# Ask a question based on the documents you loaded into the /data folder
response = query_engine.query("Summarize these documents in 5 bullet points.")

# Concept: The response is synthesized using the retrieved data to ensure 
# accuracy and contextual relevance [15, 16].
print("Response:")
print(response)

2026-02-08 20:41:42,766 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-02-08 20:41:42,848 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:42:20,155 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response:
- The documents include terms of employment for a candidate at Accenture Operations Delivery centers, India, detailing confidentiality agreements, non-disparagement clauses, and provisions regarding termination and payment settlements.
- Annexure 2 from the Offer Letter outlines required documentation for a candidate's acceptance, including recent photographs, PAN card copy, educational certifications, age-related documents, relieving letters, employment dates, Aadhaar Card copies, UAN numbers, PF statements, and Forms for tax purposes.
- These materials emphasize confidentiality of information during employment and upon termination or expiration of employment, as well as the candidate's responsibility to update professional references and social media accounts.


## Step 6: Build Chatbot

In [34]:
def doc_chat(question: str):
    prompt = (
        "You are an assistant that answers ONLY using the loaded documents. "
        "Always answer concisely and focus on factual details from the documents.\n\n"
        f"Question: {question}"
    )
    return query_engine.query(prompt)


In [35]:
while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit"}:
        break
    print(f'User:{q}')
    resp = doc_chat(q)
    print("\nBot:", resp, "\n")

User:Provide the condidate name as per released offer?


2026-02-08 20:44:06,966 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:44:20,279 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



Bot: Pawan Kumar 

User:and condidate ID?


2026-02-08 20:44:39,603 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:45:02,228 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



Bot: The provided context does not contain any information about a candidate's ID. 



## Step 7: Building the Agentic RAG Pipeline
We wrap the index in a QueryEngineTool, allowing a ReAct Agent to use it autonomously. This agent can "think" and decide when to retrieve data

In [36]:
from llama_index.core.tools import QueryEngineTool

query_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="local_knowledge_search",
    description="Search within the ingested private offer/HR documents.",
    return_direct=False,
)

In [37]:
import inspect
from llama_index.core.tools import QueryEngineTool
print(inspect.signature(QueryEngineTool.__init__))
print(inspect.signature(QueryEngineTool.from_defaults))

(self, query_engine: llama_index.core.base.base_query_engine.BaseQueryEngine, metadata: llama_index.core.tools.types.ToolMetadata, resolve_input_errors: bool = True) -> None
(query_engine: llama_index.core.base.base_query_engine.BaseQueryEngine, name: Optional[str] = None, description: Optional[str] = None, return_direct: bool = False, resolve_input_errors: bool = True) -> 'QueryEngineTool'


In [38]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core import Settings

agent = ReActAgent(
    tools=[query_tool],
    llm=Settings.llm,
    verbose=False,
)

ctx = Context(agent)

In [39]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

async def chat_with_agent(q: str):
    handler = agent.run(
        user_msg=q,
        ctx=ctx,
        max_iterations=4,
        early_stopping_method="generate",
        timeout=180,
    )
    result = await handler
    return str(result)

while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit"}:
        break
    resp = asyncio.run(chat_with_agent(q))  # ← now works with nest_asyncio
    print("Bot:", resp)

2026-02-08 20:45:52,082 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-02-08 20:46:05,160 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:46:18,491 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-02-08 20:46:22,652 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Bot: The candidate ID can only be determined by referring to the released offer document. However, as per the information given, it seems like a tool for searching within the HR documents (local_knowledge_search) is not able to provide this detail. Please refer to the released offer document for the Candidate ID.


2026-02-08 20:47:24,594 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-02-08 20:47:52,519 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-02-08 20:48:31,973 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-02-08 20:48:37,849 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Bot: The required documents are as follows:
1. Two passport size copies of recent photographs
2. PAN card copy
3. Copy of highest education certificates
4. Copy of last semester's mark sheets
5. Documents in support of age (such as 10th/12th Marksheet or Passport Copy)
6. Relieving letters from previous employers
7. Document/s containing start and end dates for the last two employers
8. Passport copy, if available
9. UAN Number and PF Statement for the last two employments before Accenture
10. Form 16 and Form 26AS from the last two employments before Accenture
11. ESIC card or Form 1 Declaration if eligible
12. Aadhaar Card - For meeting UAN generation requirement and compliance with regulatory authorities like EPFO, ESIC, labor welfare fund
